# Medical Diagnosis Using Large Language Models and Retrieval-Augmented Generation

## Business Context

Healthcare professionals must navigate large volumes of medical information while making timely and accurate decisions. A standalone large language model can generate fluent answers, but its responses may be incomplete, unsupported, or inconsistent with the approved medical reference.

This project develops and evaluates a **Retrieval-Augmented Generation (RAG)** prototype using *The Merck Manual of Diagnosis and Therapy, 19th Edition*. The system retrieves relevant passages from the manual and supplies them to an instruction-tuned language model before generating an answer.

> **Scope and safety:** This notebook is an educational clinical-information retrieval prototype. It does not diagnose patients, prescribe treatment, or replace assessment by a qualified healthcare professional.

## Objective

The objectives are to:

1. Generate baseline answers using a standalone Hugging Face large language model.
2. apply at least five prompt-engineering and generation-parameter combinations;
3. prepare the Merck Manual for RAG through loading, cleaning, chunking, embedding, and vector indexing;
4. generate RAG answers and compare at least five chunking, retrieval, and LLM configurations;
5. evaluate the final answers for groundedness and relevance; and
6. translate the technical findings into actionable healthcare recommendations.

## Data Description

The supplied dataset is a PDF of *The Merck Manual of Diagnosis and Therapy, 19th Edition*. It contains more than 4,000 PDF pages across 23 medical sections, including gastrointestinal disorders, dermatology, neurology, critical care medicine, injuries, and poisoning.

The five assignment questions concern:

- sepsis and septic shock;
- appendicitis;
- sudden patchy hair loss;
- traumatic brain injury; and
- leg fracture management during a hiking emergency.

## Runtime Requirement

In Google Colab, select:

**Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**

The next cell verifies GPU availability.

In [ ]:
!nvidia-smi

## Installing and Importing Required Libraries

In [ ]:
# GPU-enabled llama-cpp-python and supporting RAG libraries.
# The first installation can take several minutes.
!CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install -q --upgrade --force-reinstall llama-cpp-python==0.2.90
!pip install -q \
    huggingface-hub==0.34.4 \
    pandas==2.2.2 \
    numpy==1.26.4 \
    tiktoken==0.11.0 \
    pymupdf==1.26.3 \
    langchain==0.3.27 \
    langchain-community==0.3.27 \
    chromadb==1.0.20 \
    sentence-transformers==5.1.0

print("Installation completed.")
print("If Colab asks for a runtime restart, restart once and rerun the notebook from this cell.")

In [ ]:
import os
import re
import gc
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tiktoken

from huggingface_hub import hf_hub_download
from llama_cpp import Llama

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 160)

print("Libraries imported successfully.")

## Assignment Questions

In [ ]:
questions = [
    "What is the protocol for managing sepsis in a critical care unit?",
    "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
    "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
]

question_ids = [f"Q{i}" for i in range(1, len(questions) + 1)]
pd.DataFrame({"Question ID": question_ids, "Question": questions})

# 1. Question Answering Using a Standalone LLM

A quantized instruction-tuned Mistral model is downloaded from Hugging Face and loaded locally. The baseline model does not receive any text from the Merck Manual. Its answers therefore reflect only its pretrained knowledge and the wording of the question.

### Downloading and Loading the Hugging Face Model

In [ ]:
MODEL_REPO = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
MODEL_FILE = "mistral-7b-instruct-v0.2.Q4_K_M.gguf"

model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE
)

print("Model downloaded to:", model_path)

In [ ]:
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_gpu_layers=-1,      # offload all possible layers to the T4 GPU
    n_batch=512,
    verbose=False,
    seed=42
)

print("LLM loaded successfully.")

### Reusable Response-Generation Function

In [ ]:
def format_mistral_prompt(system_message: str, user_message: str) -> str:
    """Format a prompt for Mistral Instruct."""
    return (
        "<s>[INST] "
        + system_message.strip()
        + "\n\n"
        + user_message.strip()
        + " [/INST]"
    )


def generate_response(
    user_message: str,
    system_message: str = "You are a helpful medical information assistant.",
    max_tokens: int = 320,
    temperature: float = 0.0,
    top_p: float = 0.95,
    top_k: int = 40,
    repeat_penalty: float = 1.1,
    seed: int = 42
) -> str:
    """Generate a response using the local instruction-tuned LLM."""
    prompt = format_mistral_prompt(system_message, user_message)

    output = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repeat_penalty=repeat_penalty,
        seed=seed,
        stop=["</s>", "[INST]"]
    )
    return output["choices"][0]["text"].strip()

### Baseline Answers to All Five Questions

In [ ]:
baseline_rows = []

for qid, question in zip(question_ids, questions):
    start = time.time()
    answer = generate_response(
        user_message=question,
        system_message="You are a helpful medical information assistant.",
        max_tokens=320,
        temperature=0.0,
        top_p=0.95,
        top_k=40
    )
    baseline_rows.append({
        "question_id": qid,
        "question": question,
        "answer": answer,
        "word_count": len(answer.split()),
        "runtime_seconds": round(time.time() - start, 2)
    })
    print(f"\n{'='*100}\n{qid}: {question}\n{'-'*100}\n{answer}")

baseline_df = pd.DataFrame(baseline_rows)
baseline_df

### Observations — Standalone LLM

- The baseline establishes how the model answers without access to the supplied medical manual.
- Fluency or clinical tone alone does not prove factual reliability. Any treatment, dosage, timing, or surgical recommendation may be unsupported because no reference passage was supplied.
- The answers should be reviewed for completeness across every part of each question, especially the causes and treatment of patchy hair loss and the immediate precautions, definitive treatment, and recovery considerations for a hiking-related fracture.
- These limitations motivate prompt engineering and, more importantly, source-grounded retrieval.

# 2. Question Answering Using LLM with Prompt Engineering

Five prompt/parameter combinations are tested on **all five questions**. This gives 25 prompt-engineered responses and meets the rubric requirement while preserving a fair comparison.

The experiments vary:

- role definition;
- required answer structure;
- safety and uncertainty instructions;
- temperature, top-p, top-k, and response length.

In [ ]:
prompt_experiments = [
    {
        "experiment": "PE1_Direct",
        "system": "You are a medical information assistant. Answer clearly and concisely.",
        "instruction": "Answer the following question directly.",
        "temperature": 0.0, "top_p": 0.95, "top_k": 40, "max_tokens": 300
    },
    {
        "experiment": "PE2_Clinical_Role",
        "system": (
            "You are a careful clinical knowledge assistant supporting healthcare professionals. "
            "Use professional terminology, distinguish emergency care from definitive treatment, "
            "and do not claim certainty when information is missing."
        ),
        "instruction": "Provide an evidence-oriented clinical explanation.",
        "temperature": 0.1, "top_p": 0.90, "top_k": 30, "max_tokens": 360
    },
    {
        "experiment": "PE3_Structured",
        "system": (
            "You are a medical information assistant. Organize the answer under relevant headings: "
            "likely condition, symptoms or warning signs, immediate management, definitive treatment, "
            "precautions, and follow-up. Include only headings that apply."
        ),
        "instruction": "Give a structured answer using short bullets.",
        "temperature": 0.2, "top_p": 0.90, "top_k": 40, "max_tokens": 400
    },
    {
        "experiment": "PE4_Conservative_Safety",
        "system": (
            "You are a conservative medical information assistant. Avoid unsupported claims, "
            "avoid inventing drug doses, clearly identify urgent red flags, and state when immediate "
            "professional or emergency care is required."
        ),
        "instruction": "Answer cautiously and explicitly identify safety-critical actions.",
        "temperature": 0.0, "top_p": 0.80, "top_k": 20, "max_tokens": 380
    },
    {
        "experiment": "PE5_Comprehensive",
        "system": (
            "You are a senior clinical decision-support assistant. Give a concise but comprehensive "
            "answer covering causes, presentation, assessment, immediate care, definitive management, "
            "contraindications or precautions, and recovery where relevant. This is educational "
            "information and not a substitute for clinician assessment."
        ),
        "instruction": "Address every component of the question and use numbered steps where useful.",
        "temperature": 0.15, "top_p": 0.92, "top_k": 50, "max_tokens": 450
    }
]

pd.DataFrame(prompt_experiments)[
    ["experiment", "temperature", "top_p", "top_k", "max_tokens"]
]

In [ ]:
prompt_rows = []

for config in prompt_experiments:
    for qid, question in zip(question_ids, questions):
        answer = generate_response(
            user_message=f"{config['instruction']}\n\nQuestion: {question}",
            system_message=config["system"],
            max_tokens=config["max_tokens"],
            temperature=config["temperature"],
            top_p=config["top_p"],
            top_k=config["top_k"]
        )
        prompt_rows.append({
            "experiment": config["experiment"],
            "question_id": qid,
            "question": question,
            "answer": answer,
            "word_count": len(answer.split())
        })
        print(
            f"\n{'='*100}\n{config['experiment']} | {qid}\n"
            f"{question}\n{'-'*100}\n{answer}"
        )

prompt_results_df = pd.DataFrame(prompt_rows)
prompt_results_df

In [ ]:
# Compact comparison of response length across the five prompt experiments
prompt_summary_df = (
    prompt_results_df
    .groupby("experiment", as_index=False)
    .agg(
        responses=("answer", "count"),
        average_words=("word_count", "mean"),
        minimum_words=("word_count", "min"),
        maximum_words=("word_count", "max")
    )
    .round(1)
)

prompt_summary_df

### Observations — Prompt Engineering

- A direct prompt provides a useful baseline but may omit parts of compound questions.
- The clinical-role prompt should improve professional focus, while the structured prompt makes symptoms, immediate actions, treatment, and follow-up easier to inspect.
- The conservative safety prompt is expected to reduce speculation and unsupported dosing, although it may produce shorter answers.
- The comprehensive prompt is likely to improve coverage but can also increase verbosity and the number of claims that require verification.
- Prompt engineering improves presentation and caution, but it still does not connect the response to the supplied Merck Manual. RAG is therefore required for source traceability and groundedness.

# 3. Data Preparation for RAG

## 3.1 Loading the Merck Manual PDF

In [ ]:
# The notebook first looks for the standard Colab path.
# Upload the PDF to Colab with the exact name: medical_diagnosis_manual.pdf

PDF_CANDIDATES = [
    "/content/medical_diagnosis_manual.pdf",
    "/content/drive/MyDrive/medical_diagnosis_manual.pdf"
]

pdf_path = next((p for p in PDF_CANDIDATES if os.path.exists(p)), None)

if pdf_path is None:
    from google.colab import files
    print("Please upload medical_diagnosis_manual.pdf")
    uploaded = files.upload()
    matching = [name for name in uploaded if name.lower().endswith(".pdf")]
    if not matching:
        raise FileNotFoundError("No PDF file was uploaded.")
    pdf_path = f"/content/{matching[0]}"

print("Using PDF:", pdf_path)

loader = PyMuPDFLoader(pdf_path)
manual_pages = loader.load()

print("Number of loaded PDF pages:", len(manual_pages))

## 3.2 Data Overview and Quality Checks

In [ ]:
for i in range(min(5, len(manual_pages))):
    text = manual_pages[i].page_content.strip()
    print(f"\n--- PDF Page {i + 1} ---")
    print(text[:700] if text else "[No extractable text]")

In [ ]:
page_lengths = [len(doc.page_content.strip()) for doc in manual_pages]

data_quality_summary = pd.DataFrame({
    "metric": [
        "Total pages",
        "Blank or near-blank pages (<50 characters)",
        "Median characters per page",
        "Maximum characters on a page"
    ],
    "value": [
        len(manual_pages),
        sum(length < 50 for length in page_lengths),
        int(np.median(page_lengths)),
        max(page_lengths)
    ]
})

data_quality_summary

### Observation — Data Loading

The manual is sufficiently large to make manual search inefficient and to justify a retrieval layer. PDF extraction can retain repeated ownership notices, headers, footers, and page labels. These repeated elements are removed before chunking because they add no medical meaning and can reduce retrieval quality.

## 3.3 Cleaning Repeated PDF Text

In [ ]:
WATERMARK_PATTERNS = [
    r"click2shivesh@gmail\.com",
    r"WPFSN4D3ZX",
    r"This file is meant for personal use by .*? only\.",
    r"Sharing or publishing the contents in part or full is liable for legal action\."
]

def clean_page_text(text: str) -> str:
    cleaned = text
    for pattern in WATERMARK_PATTERNS:
        cleaned = re.sub(pattern, " ", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

clean_pages = []
for doc in manual_pages:
    cleaned_text = clean_page_text(doc.page_content)
    if len(cleaned_text) >= 50:
        doc.page_content = cleaned_text
        # PyMuPDFLoader uses zero-based page metadata; keep both machine and reader-friendly page values.
        zero_based_page = int(doc.metadata.get("page", 0))
        doc.metadata["pdf_page"] = zero_based_page + 1
        clean_pages.append(doc)

print("Pages retained after cleaning:", len(clean_pages))
print("Sample cleaned text:\\n", clean_pages[0].page_content[:700])

## 3.4 Text Chunking

In [ ]:
BASE_CHUNK_SIZE = 1000
BASE_CHUNK_OVERLAP = 200

base_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=BASE_CHUNK_SIZE,
    chunk_overlap=BASE_CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

base_chunks = base_splitter.split_documents(clean_pages)

for idx, chunk in enumerate(base_chunks):
    chunk.metadata["chunk_id"] = idx

print("Number of chunks:", len(base_chunks))
print("First chunk metadata:", base_chunks[0].metadata)
print("First chunk preview:\n", base_chunks[0].page_content[:700])

### Observation — Chunking

A 1,000-token chunk with 200-token overlap is used as the baseline. The overlap reduces the risk that a symptom, diagnosis, or treatment sequence is separated at a chunk boundary. Very small chunks may lose clinical context, while very large chunks can introduce unrelated material and consume the LLM context window.

## 3.5 Embedding Model

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

sample_embedding = embedding_model.embed_query(
    "management of sepsis and septic shock"
)

print("Embedding dimension:", len(sample_embedding))
print("First 10 values:", sample_embedding[:10])

### Observation — Embeddings

`all-MiniLM-L6-v2` produces compact 384-dimensional semantic embeddings. Normalisation supports meaningful cosine-style similarity comparisons. The model offers an appropriate balance of retrieval quality, memory use, and processing time for a 4,000-page educational prototype.

## 3.6 Vector Database

In [ ]:
BASE_DB_DIR = "/content/medical_chroma_base"

# Delete an old partial index only when intentionally rebuilding.
# import shutil
# shutil.rmtree(BASE_DB_DIR, ignore_errors=True)

if os.path.exists(BASE_DB_DIR) and os.listdir(BASE_DB_DIR):
    vectorstore = Chroma(
        persist_directory=BASE_DB_DIR,
        embedding_function=embedding_model,
        collection_name="merck_manual_base"
    )
    print("Loaded existing Chroma index.")
else:
    vectorstore = Chroma.from_documents(
        documents=base_chunks,
        embedding=embedding_model,
        persist_directory=BASE_DB_DIR,
        collection_name="merck_manual_base"
    )
    print("Created and persisted a new Chroma index.")

print("Vector database is ready.")

## 3.7 Retriever

In [ ]:
baseline_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": 0.7
    }
)

test_docs = baseline_retriever.invoke(
    "protocol for management of sepsis and septic shock in critical care"
)

for rank, doc in enumerate(test_docs, start=1):
    print(
        f"Rank {rank} | PDF page {doc.metadata.get('pdf_page')} | "
        f"Chunk {doc.metadata.get('chunk_id')}"
    )
    print(doc.page_content[:500], "\n")

### Observation — Retriever

Maximum Marginal Relevance (MMR) is used for the baseline retriever because it balances relevance with diversity. This is useful for multi-part medical questions where symptoms, emergency management, definitive treatment, and recovery may appear in different passages.

# 4. Question Answering Using RAG

## 4.1 RAG Prompt and Helper Functions

In [ ]:
RAG_SYSTEM_MESSAGE = """
You are a clinical knowledge assistant using The Merck Manual.

Rules:
1. Use only the supplied context.
2. Do not add facts, doses, timings, or recommendations that are absent from the context.
3. If the context does not support part of the answer, explicitly state that the retrieved context is insufficient.
4. Address every component of the question.
5. Separate immediate precautions, assessment, definitive treatment, and recovery where relevant.
6. Cite supporting PDF pages in square brackets, for example [PDF page 2450].
7. This is educational decision-support information and does not replace qualified clinical assessment.
""".strip()

RAG_USER_TEMPLATE = """
CONTEXT FROM THE MERCK MANUAL:
{context}

QUESTION:
{question}

Produce a concise, structured, context-grounded answer.
""".strip()


QUERY_HINTS = {
    questions[0]: "sepsis septic shock critical care diagnosis cultures antibiotics fluid resuscitation vasopressors",
    questions[1]: "appendicitis symptoms diagnosis antibiotics appendectomy laparoscopic surgery acute abdomen",
    questions[2]: "alopecia areata patchy hair loss causes treatment corticosteroids minoxidil tinea capitis",
    questions[3]: "traumatic brain injury treatment intracranial pressure monitoring surgery rehabilitation",
    questions[4]: "leg fracture wilderness first aid immobilization splint neurovascular status reduction surgery rehabilitation"
}


def retrieve_documents(
    question: str,
    retriever,
    k: int
):
    expanded_query = question + "\nRelevant medical concepts: " + QUERY_HINTS.get(question, "")
    docs = retriever.invoke(expanded_query)
    return docs[:k]


def build_context(docs, max_context_tokens: int = 2600) -> str:
    enc = tiktoken.get_encoding("cl100k_base")
    sections = []
    used = 0

    for rank, doc in enumerate(docs, start=1):
        page = doc.metadata.get("pdf_page", "unknown")
        chunk_id = doc.metadata.get("chunk_id", "unknown")
        block = (
            f"[SOURCE {rank} | PDF page {page} | chunk {chunk_id}]\n"
            f"{doc.page_content.strip()}"
        )
        token_ids = enc.encode(block)

        remaining = max_context_tokens - used
        if remaining <= 0:
            break

        if len(token_ids) > remaining:
            block = enc.decode(token_ids[:remaining])
            sections.append(block)
            break

        sections.append(block)
        used += len(token_ids)

    return "\n\n---\n\n".join(sections)


def generate_rag_response(
    question: str,
    retriever=baseline_retriever,
    k: int = 5,
    max_context_tokens: int = 2600,
    max_tokens: int = 450,
    temperature: float = 0.0,
    top_p: float = 0.90,
    top_k: int = 30
):
    docs = retrieve_documents(question, retriever, k)
    context = build_context(docs, max_context_tokens=max_context_tokens)

    user_message = RAG_USER_TEMPLATE.format(
        context=context,
        question=question
    )

    answer = generate_response(
        user_message=user_message,
        system_message=RAG_SYSTEM_MESSAGE,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )

    source_pages = sorted({
        doc.metadata.get("pdf_page")
        for doc in docs
        if doc.metadata.get("pdf_page") is not None
    })

    return {
        "answer": answer,
        "context": context,
        "documents": docs,
        "source_pages": source_pages
    }

## 4.2 Baseline RAG Answers to All Five Questions

In [ ]:
baseline_rag_rows = []
baseline_rag_details = {}

for qid, question in zip(question_ids, questions):
    result = generate_rag_response(
        question=question,
        retriever=baseline_retriever,
        k=5,
        max_context_tokens=2600,
        max_tokens=450,
        temperature=0.0,
        top_p=0.90,
        top_k=30
    )

    baseline_rag_details[qid] = result
    baseline_rag_rows.append({
        "question_id": qid,
        "question": question,
        "answer": result["answer"],
        "source_pages": result["source_pages"],
        "word_count": len(result["answer"].split())
    })

    print(
        f"\n{'='*100}\n{qid}: {question}\n"
        f"Retrieved PDF pages: {result['source_pages']}\n"
        f"{'-'*100}\n{result['answer']}"
    )

baseline_rag_df = pd.DataFrame(baseline_rag_rows)
baseline_rag_df

### Observations — Baseline RAG

- Each answer is generated from retrieved Merck Manual passages rather than from the model’s memory alone.
- PDF-page metadata makes the result auditable and allows a reviewer to inspect the supporting passage.
- The instruction to acknowledge insufficient context is important: an incomplete but transparent answer is safer than a complete-sounding unsupported answer.
- Retrieval quality remains the key dependency. A well-written answer can still miss information when the retriever selects the wrong chapter or too few chunks.

# 5. Fine-Tuning RAG Parameters — Five Combinations

The following experiments vary all three areas required by the rubric:

1. **Chunking:** chunk size and overlap;
2. **Retriever:** similarity versus MMR and different `k` values; and
3. **LLM:** temperature, top-p, top-k, and maximum answer length.

To keep Colab memory stable, each experiment is built, tested on all five questions, saved, and then released before the next experiment. The final selected configuration is rebuilt afterward.

In [ ]:
rag_experiments = [
    {
        "experiment": "RAG1_Small_Similarity",
        "chunk_size": 600, "chunk_overlap": 100,
        "search_type": "similarity", "k": 3, "fetch_k": None, "lambda_mult": None,
        "temperature": 0.0, "top_p": 0.90, "top_k": 30, "max_tokens": 380
    },
    {
        "experiment": "RAG2_Medium_Similarity",
        "chunk_size": 800, "chunk_overlap": 150,
        "search_type": "similarity", "k": 5, "fetch_k": None, "lambda_mult": None,
        "temperature": 0.1, "top_p": 0.90, "top_k": 30, "max_tokens": 420
    },
    {
        "experiment": "RAG3_Balanced_MMR",
        "chunk_size": 1000, "chunk_overlap": 200,
        "search_type": "mmr", "k": 5, "fetch_k": 20, "lambda_mult": 0.7,
        "temperature": 0.0, "top_p": 0.90, "top_k": 30, "max_tokens": 450
    },
    {
        "experiment": "RAG4_Large_MMR",
        "chunk_size": 1200, "chunk_overlap": 250,
        "search_type": "mmr", "k": 6, "fetch_k": 24, "lambda_mult": 0.6,
        "temperature": 0.1, "top_p": 0.92, "top_k": 40, "max_tokens": 480
    },
    {
        "experiment": "RAG5_Wide_MMR",
        "chunk_size": 1500, "chunk_overlap": 300,
        "search_type": "mmr", "k": 8, "fetch_k": 32, "lambda_mult": 0.5,
        "temperature": 0.0, "top_p": 0.85, "top_k": 20, "max_tokens": 500
    }
]

pd.DataFrame(rag_experiments)

In [ ]:
def build_experiment_vectorstore(config):
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        encoding_name="cl100k_base",
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = splitter.split_documents(clean_pages)

    for idx, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = idx

    collection_name = re.sub(r"[^a-zA-Z0-9_-]", "_", config["experiment"]).lower()
    persist_dir = f"/content/{collection_name}_db"

    if os.path.exists(persist_dir) and os.listdir(persist_dir):
        store = Chroma(
            persist_directory=persist_dir,
            embedding_function=embedding_model,
            collection_name=collection_name
        )
    else:
        store = Chroma.from_documents(
            documents=chunks,
            embedding=embedding_model,
            persist_directory=persist_dir,
            collection_name=collection_name
        )

    if config["search_type"] == "mmr":
        retriever = store.as_retriever(
            search_type="mmr",
            search_kwargs={
                "k": config["k"],
                "fetch_k": config["fetch_k"],
                "lambda_mult": config["lambda_mult"]
            }
        )
    else:
        retriever = store.as_retriever(
            search_type="similarity",
            search_kwargs={"k": config["k"]}
        )

    return chunks, store, retriever

In [ ]:
rag_experiment_rows = []
rag_experiment_details = {}

for config in rag_experiments:
    print(f"\nBuilding {config['experiment']} ...")
    exp_chunks, exp_store, exp_retriever = build_experiment_vectorstore(config)
    print("Chunks:", len(exp_chunks))

    for qid, question in zip(question_ids, questions):
        result = generate_rag_response(
            question=question,
            retriever=exp_retriever,
            k=config["k"],
            max_context_tokens=2700,
            max_tokens=config["max_tokens"],
            temperature=config["temperature"],
            top_p=config["top_p"],
            top_k=config["top_k"]
        )

        rag_experiment_details[(config["experiment"], qid)] = result

        rag_experiment_rows.append({
            "experiment": config["experiment"],
            "question_id": qid,
            "chunk_size": config["chunk_size"],
            "chunk_overlap": config["chunk_overlap"],
            "search_type": config["search_type"],
            "k": config["k"],
            "temperature": config["temperature"],
            "answer": result["answer"],
            "source_pages": result["source_pages"],
            "word_count": len(result["answer"].split())
        })

        print(
            f"\n{config['experiment']} | {qid} | "
            f"pages={result['source_pages']}\n{result['answer']}"
        )

    # Release the active experiment objects before building the next index.
    del exp_chunks, exp_retriever, exp_store
    gc.collect()

rag_experiment_df = pd.DataFrame(rag_experiment_rows)
rag_experiment_df

In [ ]:
rag_parameter_summary = (
    rag_experiment_df
    .groupby(
        ["experiment", "chunk_size", "chunk_overlap", "search_type", "k", "temperature"],
        as_index=False
    )
    .agg(
        responses=("answer", "count"),
        average_words=("word_count", "mean"),
        average_source_pages=("source_pages", lambda x: np.mean([len(v) for v in x]))
    )
    .round(2)
)

rag_parameter_summary

### Observations — RAG Parameter Tuning

- Smaller chunks can retrieve a highly focused passage but may separate a clinical sequence across multiple chunks.
- Larger chunks preserve context but can include unrelated material and leave less room for generation.
- Similarity search prioritises the nearest passages; MMR can improve coverage by reducing near-duplicate results.
- Increasing `k` can improve completeness for multi-part questions, but excessive `k` may add noise.
- Low temperature is appropriate for medical reference summarisation because consistency and fidelity are more important than creativity.
- The final configuration should be selected from the observed retrieval quality, answer completeness, groundedness, and relevance—not from response length alone.

# 6. Final Configuration and Answers

The balanced MMR configuration is used as the initial final candidate:

- chunk size: 1,000 tokens;
- overlap: 200 tokens;
- MMR retrieval;
- `k = 5`;
- temperature: 0.0.

After execution, the evaluation table below should be used to confirm or change this selection.

In [ ]:
FINAL_EXPERIMENT = "RAG3_Balanced_MMR"
final_config = next(
    config for config in rag_experiments
    if config["experiment"] == FINAL_EXPERIMENT
)

final_chunks, final_vectorstore, final_retriever = build_experiment_vectorstore(final_config)

print("Selected configuration:")
display(pd.DataFrame([final_config]))
print("Final number of chunks:", len(final_chunks))

In [ ]:
final_rows = []
final_details = {}

for qid, question in zip(question_ids, questions):
    result = generate_rag_response(
        question=question,
        retriever=final_retriever,
        k=final_config["k"],
        max_context_tokens=2700,
        max_tokens=final_config["max_tokens"],
        temperature=final_config["temperature"],
        top_p=final_config["top_p"],
        top_k=final_config["top_k"]
    )

    final_details[qid] = result
    final_rows.append({
        "question_id": qid,
        "question": question,
        "final_rag_answer": result["answer"],
        "source_pages": result["source_pages"]
    })

    print(
        f"\n{'='*100}\nFINAL {qid}: {question}\n"
        f"Retrieved PDF pages: {result['source_pages']}\n"
        f"{'-'*100}\n{result['answer']}"
    )

final_answers_df = pd.DataFrame(final_rows)
final_answers_df

# 7. Output Evaluation

The final RAG responses are evaluated on two separate dimensions using an LLM-as-a-judge approach:

- **Groundedness:** whether the important claims in the answer are supported by the retrieved context.
- **Relevance:** whether the answer directly and completely addresses the question.

The evaluator is instructed to return structured JSON with a score from 1 to 5. Because the same local model acts as generator and judge, these scores are useful for consistent prototype comparison but are not equivalent to independent expert clinical validation.

## 7.1 Groundedness and Relevance Evaluation Prompts

In [ ]:
GROUNDEDNESS_SYSTEM = """
You are a strict evaluator of medical-answer groundedness.

Compare the ANSWER only with the supplied CONTEXT.
Do not reward general medical plausibility.
Penalize any diagnosis, treatment, dose, timing, cause, prognosis, or precaution
that is not supported by the context.

Score:
1 = mostly unsupported or contradictory
2 = several major unsupported claims
3 = partly supported, with important unsupported claims
4 = well supported, with only minor unsupported details
5 = fully supported by the context

Return valid JSON only:
{"score": <1-5>, "explanation": "<brief reason>", "unsupported_claims": ["..."]}
""".strip()


RELEVANCE_SYSTEM = """
You are a strict evaluator of answer relevance and completeness.

Compare the ANSWER with the QUESTION.
Check whether every component of the question is addressed directly and clearly.

Score:
1 = irrelevant
2 = only a small part is addressed
3 = generally relevant but materially incomplete
4 = relevant and mostly complete
5 = directly relevant, complete, and well structured

Return valid JSON only:
{"score": <1-5>, "explanation": "<brief reason>", "missing_information": ["..."]}
""".strip()


def parse_json_safely(text: str) -> dict:
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return {"score": None, "explanation": text, "issues": []}
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return {"score": None, "explanation": text, "issues": []}


def evaluate_groundedness(context: str, answer: str) -> dict:
    message = f"CONTEXT:\n{context}\n\nANSWER:\n{answer}"
    raw = generate_response(
        user_message=message,
        system_message=GROUNDEDNESS_SYSTEM,
        max_tokens=220,
        temperature=0.0,
        top_p=0.8,
        top_k=20
    )
    result = parse_json_safely(raw)
    result["raw_output"] = raw
    return result


def evaluate_relevance(question: str, answer: str) -> dict:
    message = f"QUESTION:\n{question}\n\nANSWER:\n{answer}"
    raw = generate_response(
        user_message=message,
        system_message=RELEVANCE_SYSTEM,
        max_tokens=220,
        temperature=0.0,
        top_p=0.8,
        top_k=20
    )
    result = parse_json_safely(raw)
    result["raw_output"] = raw
    return result

## 7.2 Evaluation of All Final Responses

In [ ]:
evaluation_rows = []

for qid, question in zip(question_ids, questions):
    detail = final_details[qid]
    answer = detail["answer"]
    context = detail["context"]

    groundedness = evaluate_groundedness(context, answer)
    relevance = evaluate_relevance(question, answer)

    evaluation_rows.append({
        "question_id": qid,
        "groundedness_score": groundedness.get("score"),
        "groundedness_explanation": groundedness.get("explanation"),
        "unsupported_claims": groundedness.get("unsupported_claims", []),
        "relevance_score": relevance.get("score"),
        "relevance_explanation": relevance.get("explanation"),
        "missing_information": relevance.get("missing_information", [])
    })

evaluation_df = pd.DataFrame(evaluation_rows)
evaluation_df

In [ ]:
evaluation_summary = pd.DataFrame({
    "metric": [
        "Average groundedness score",
        "Average relevance score",
        "Responses evaluated"
    ],
    "value": [
        pd.to_numeric(evaluation_df["groundedness_score"], errors="coerce").mean(),
        pd.to_numeric(evaluation_df["relevance_score"], errors="coerce").mean(),
        len(evaluation_df)
    ]
})

evaluation_summary

### Observations — Output Evaluation

- Groundedness and relevance measure different properties. An answer may be fully supported yet incomplete, or highly relevant yet contain unsupported details.
- Unsupported-claim lists help identify exactly where prompt restrictions, retrieval, or context size require improvement.
- Missing-information lists indicate whether the retriever should retrieve more diverse chunks or whether the prompt should more explicitly request every part of the question.
- Final model selection should prioritise groundedness first, followed by relevance and clarity, because an unsupported clinical statement is more serious than a concise or stylistic limitation.
- LLM-as-a-judge results should be supplemented with expert review before any real healthcare deployment.

# 8. Comparison of Standalone LLM, Prompt Engineering, and RAG

In [ ]:
comparison_df = pd.DataFrame({
    "Approach": [
        "Standalone LLM",
        "Prompt-engineered LLM",
        "RAG using the Merck Manual"
    ],
    "Primary strength": [
        "Fast and fluent general response",
        "Better structure, caution, and task coverage",
        "Source-grounded and traceable answer"
    ],
    "Primary limitation": [
        "No supplied reference and higher hallucination risk",
        "Prompting cannot guarantee factual grounding",
        "Depends on document quality and retrieval accuracy"
    ],
    "Recommended role": [
        "Exploratory baseline only",
        "Improved baseline and prompt testing",
        "Preferred educational decision-support prototype"
    ]
})

comparison_df

# 9. Actionable Insights and Business Recommendations

## Key Takeaways

### 1. Reduce information-search time

A RAG interface can retrieve relevant passages from thousands of manual pages within seconds. This can reduce the time spent navigating indexes and chapters, particularly for uncommon or multi-part clinical questions.

### 2. Improve traceability and accountability

Every generated answer should display the supporting manual pages and retrieved excerpts. Source visibility allows healthcare professionals to verify the system’s interpretation rather than accepting an opaque answer.

### 3. Standardise access to approved knowledge

Using a centrally governed reference can help different teams receive more consistent information. The same validated knowledge base, prompt template, and retrieval configuration can be deployed across departments.

### 4. Keep clinicians in control

The prototype should support information retrieval and summarisation—not autonomous diagnosis or treatment. Clinical judgement, patient history, examination, investigations, contraindications, and current local protocols remain essential.

### 5. Introduce medical-content governance

A production system should include:

- clinician-approved source documents;
- document and model versioning;
- visible publication or revision dates;
- scheduled knowledge-base updates;
- audit logs of questions, retrieved passages, and answers;
- role-based access controls;
- privacy and security safeguards;
- automated detection of unsupported statements; and
- escalation when context is insufficient.

### 6. Address the age of the source

The supplied PDF is the 19th edition from 2011. It is appropriate for demonstrating RAG feasibility, but a real clinical system must use current, approved manuals, local protocols, drug references, and guidelines. Outdated source material can produce a grounded answer that is nevertheless no longer clinically current.

### 7. Validate with domain experts

Before deployment, clinicians should review:

- retrieval accuracy by specialty;
- completeness of emergency-management answers;
- medication and dosage handling;
- contraindications and special populations;
- citation correctness;
- failure cases and abstention behaviour; and
- differences between the manual and current institutional policy.

## Recommended Implementation Roadmap

1. Run a controlled pilot using non-identifiable educational queries.
2. Measure retrieval precision, groundedness, relevance, latency, and clinician satisfaction.
3. Add reranking and section-aware retrieval for complex questions.
4. Integrate current, approved sources and local clinical protocols.
5. Complete clinical safety, privacy, security, and governance review.
6. Deploy only as clinician-supervised decision support with ongoing monitoring.

# 10. Limitations and Future Improvements

## Limitations

- The supplied Merck Manual edition is dated 2011 and may not reflect current standards.
- PDF extraction may omit tables, diagrams, or formatting that carries clinical meaning.
- Dense semantic retrieval can miss exact terminology or retrieve a neighbouring but incorrect topic.
- The local LLM can still misinterpret retrieved text or generate unsupported bridging statements.
- LLM-as-a-judge evaluation is not independent clinical validation.
- The prototype does not use patient-specific history, examination findings, laboratory results, allergies, or contraindications.

## Future Improvements

- Use current and institution-approved medical sources.
- Combine semantic and keyword retrieval.
- Add a cross-encoder reranker.
- Use chapter and section metadata during retrieval.
- Extract tables and figures with layout-aware document processing.
- Add claim-level citations rather than page-level citations only.
- Introduce confidence thresholds and automatic abstention.
- Evaluate against clinician-authored reference answers.
- Test performance across specialties, age groups, and emergency scenarios.
- Monitor retrieval drift and answer quality after every knowledge-base update.

# 11. Conclusion

The project demonstrates a clear progression from a standalone LLM to prompt engineering and finally to Retrieval-Augmented Generation.

The standalone model can produce fluent medical responses, and prompt engineering can improve organisation, caution, and completeness. However, neither approach guarantees that the answer is supported by the supplied medical reference. RAG addresses this limitation by retrieving relevant Merck Manual passages before generation and exposing the source pages for verification.

The resulting prototype is best positioned as a **medical knowledge retrieval and clinical decision-support assistant**, not an autonomous diagnostic system. Its practical value lies in faster access to approved information, improved traceability, and more consistent knowledge support. Real-world use would require current clinical sources, expert validation, strong governance, privacy controls, and continuous monitoring.

## Save Key Results

The following cell saves the principal experiment and evaluation tables so they can be downloaded or reviewed later.

In [ ]:
OUTPUT_DIR = "/content/medical_rag_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

baseline_df.to_csv(f"{OUTPUT_DIR}/baseline_llm_answers.csv", index=False)
prompt_results_df.to_csv(f"{OUTPUT_DIR}/prompt_engineering_answers.csv", index=False)
rag_experiment_df.to_csv(f"{OUTPUT_DIR}/rag_experiment_answers.csv", index=False)
final_answers_df.to_csv(f"{OUTPUT_DIR}/final_rag_answers.csv", index=False)
evaluation_df.to_csv(f"{OUTPUT_DIR}/final_evaluation.csv", index=False)

print("Results saved to:", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))